In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("CS675SupplyChainJoin")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.1.2


In [2]:
inventory_path = "/home/jovyan/work/CS675_Supply_Chain_Project/data/raw/supply_chain_dataset1.csv"
holiday_path = "/home/jovyan/work/CS675_Supply_Chain_Project/data/raw/us_holidays_2024.csv"
weather_path = "/home/jovyan/work/CS675_Supply_Chain_Project/data/raw/weather2024.csv"

In [3]:
inventory_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(inventory_path)
)

print("Inventory rows:", inventory_df.count())
inventory_df.printSchema()

Inventory rows: 91250
root
 |-- Date: date (nullable = true)
 |-- SKU_ID: string (nullable = true)
 |-- Warehouse_ID: string (nullable = true)
 |-- Supplier_ID: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Units_Sold: integer (nullable = true)
 |-- Inventory_Level: integer (nullable = true)
 |-- Supplier_Lead_Time_Days: integer (nullable = true)
 |-- Reorder_Point: integer (nullable = true)
 |-- Order_Quantity: integer (nullable = true)
 |-- Unit_Cost: double (nullable = true)
 |-- Unit_Price: double (nullable = true)
 |-- Promotion_Flag: integer (nullable = true)
 |-- Stockout_Flag: integer (nullable = true)
 |-- Demand_Forecast: double (nullable = true)



In [4]:
holiday_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(holiday_path)
)

holiday_df.show(20, truncate=False)
holiday_df.printSchema()

+----------+------------------------------------+---------------+----------------+
|date      |official_name                       |date_definition|year_established|
+----------+------------------------------------+---------------+----------------+
|2024-01-01|New Year's Day                      |fixed date     |1870            |
|2024-01-15|Birthday of Martin Luther King, Jr. |3rd Monday     |1983            |
|2024-02-19|Washington's Birthday               |3rd Monday     |1879            |
|2024-05-27|Memorial Day                        |last Monday    |1868            |
|2024-06-19|Juneteenth National Independence Day|fixed date     |2021            |
|2024-07-04|Independence Day                    |fixed date     |1870            |
|2024-09-02|Labor Day                           |1st Monday     |1894            |
|2024-10-14|Columbus Day                        |2nd Monday     |1968            |
|2024-11-11|Veterans Day                        |fixed date     |1938            |
|202

In [5]:
holiday_df = holiday_df.withColumnRenamed("date", "Date")

In [6]:
holiday_df.groupBy("Date") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+----+-----+
|Date|count|
+----+-----+
+----+-----+



In [7]:
joined_df = inventory_df.join(
    F.broadcast(holiday_df),
    on="Date",
    how="left"
)

In [8]:
joined_df = joined_df.withColumn(
    "Is_Holiday",
    F.when(F.col("official_name").isNotNull(), 1).otherwise(0)
)

joined_df = joined_df.fillna({
    "official_name": "Non-Holiday",
    "date_definition": "Not Applicable"
})

In [9]:
print("Inventory rows:", inventory_df.count())
print("Rows after holiday join:", joined_df.count())

joined_df.groupBy("Is_Holiday") \
    .count() \
    .orderBy("Is_Holiday") \
    .show()

Inventory rows: 91250
Rows after holiday join: 91250
+----------+-----+
|Is_Holiday|count|
+----------+-----+
|         0|88500|
|         1| 2750|
+----------+-----+



In [10]:
joined_df.groupBy("Is_Holiday").agg(
    F.avg("Units_Sold").alias("Average_Units_Sold"),
    F.avg("Inventory_Level").alias("Average_Inventory_Level"),
    F.avg("Demand_Forecast").alias("Average_Demand_Forecast")
).show()

+----------+------------------+-----------------------+-----------------------+
|Is_Holiday|Average_Units_Sold|Average_Inventory_Level|Average_Demand_Forecast|
+----------+------------------+-----------------------+-----------------------+
|         1| 18.87127272727273|     498.79818181818183|      18.87812363636362|
|         0|20.091333333333335|      470.6747570621469|     20.119442824858698|
+----------+------------------+-----------------------+-----------------------+



In [11]:
joined_df.groupBy("Is_Holiday").agg(
    F.avg("Promotion_Flag").alias("Promotion_Rate")
).show()

+----------+-------------------+
|Is_Holiday|     Promotion_Rate|
+----------+-------------------+
|         1|0.11163636363636363|
|         0| 0.1012768361581921|
+----------+-------------------+



In [12]:
joined_df.filter(
    F.col("Is_Holiday") == 1
).groupBy(
    "official_name"
).agg(
    F.avg("Units_Sold").alias("Average_Units_Sold"),
    F.avg("Inventory_Level").alias("Average_Inventory_Level"),
    F.avg("Demand_Forecast").alias("Average_Demand_Forecast")
).orderBy(
    F.desc("Average_Units_Sold")
).show(truncate=False)

+------------------------------------+------------------+-----------------------+-----------------------+
|official_name                       |Average_Units_Sold|Average_Inventory_Level|Average_Demand_Forecast|
+------------------------------------+------------------+-----------------------+-----------------------+
|Washington's Birthday               |28.312            |463.412                |28.1082                |
|Memorial Day                        |26.392            |460.788                |26.375320000000006     |
|Birthday of Martin Luther King, Jr. |22.752            |512.656                |22.96244               |
|Juneteenth National Independence Day|21.872            |453.0                  |21.815999999999995     |
|New Year's Day                      |20.216            |721.316                |20.19908               |
|Independence Day                    |19.636            |471.72                 |19.46260000000001      |
|Christmas Day                       |19.636  

In [14]:
joined_df.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- BroadcastHashJoin LeftOuter BuildRight (6)
      :- Scan csv  (1)
      +- BroadcastExchange (5)
         +- Project (4)
            +- Filter (3)
               +- Scan csv  (2)


(1) Scan csv 
Output [15]: [Date#17, SKU_ID#18, Warehouse_ID#19, Supplier_ID#20, Region#21, Units_Sold#22, Inventory_Level#23, Supplier_Lead_Time_Days#24, Reorder_Point#25, Order_Quantity#26, Unit_Cost#27, Unit_Price#28, Promotion_Flag#29, Stockout_Flag#30, Demand_Forecast#31]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/CS675_Supply_Chain_Project/data/raw/supply_chain_dataset1.csv]
ReadSchema: struct<Date:date,SKU_ID:string,Warehouse_ID:string,Supplier_ID:string,Region:string,Units_Sold:int,Inventory_Level:int,Supplier_Lead_Time_Days:int,Reorder_Point:int,Order_Quantity:int,Unit_Cost:double,Unit_Price:double,Promotion_Flag:int,Stockout_Flag:int,Demand_Forecast:double>

(2) Scan csv 
Output [4]: [date#68, official_name#69, d

In [15]:
weather_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(weather_path)
)

print("Weather rows:", weather_df.count())

weather_df.printSchema()
weather_df.show(10, truncate=False)

Weather rows: 306
root
 |-- STATION: string (nullable = true)
 |-- DATE: date (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- ELEVATION: double (nullable = true)
 |-- NAME: string (nullable = true)
 |-- TEMP: double (nullable = true)
 |-- TEMP_ATTRIBUTES: double (nullable = true)
 |-- DEWP: double (nullable = true)
 |-- DEWP_ATTRIBUTES: double (nullable = true)
 |-- SLP: double (nullable = true)
 |-- SLP_ATTRIBUTES: double (nullable = true)
 |-- STP: double (nullable = true)
 |-- STP_ATTRIBUTES: double (nullable = true)
 |-- VISIB: double (nullable = true)
 |-- VISIB_ATTRIBUTES: double (nullable = true)
 |-- WDSP: double (nullable = true)
 |-- WDSP_ATTRIBUTES: double (nullable = true)
 |-- MXSPD: double (nullable = true)
 |-- GUST: double (nullable = true)
 |-- MAX: double (nullable = true)
 |-- MAX_ATTRIBUTES: string (nullable = true)
 |-- MIN: double (nullable = true)
 |-- MIN_ATTRIBUTES: string (nullable = true)
 |-- PRCP: doub

In [16]:
weather_clean_df = weather_df.select(
    F.col("DATE").alias("Date"),
    "TEMP",
    "MAX",
    "MIN",
    "PRCP",
    "WDSP",
    "FRSHTT"
)

In [17]:
weather_clean_df = (
    weather_clean_df
    .withColumn(
        "TEMP",
        F.when(F.col("TEMP") >= 9999, None).otherwise(F.col("TEMP"))
    )
    .withColumn(
        "MAX",
        F.when(F.col("MAX") >= 999, None).otherwise(F.col("MAX"))
    )
    .withColumn(
        "MIN",
        F.when(F.col("MIN") >= 999, None).otherwise(F.col("MIN"))
    )
    .withColumn(
        "PRCP",
        F.when(F.col("PRCP") >= 99, None).otherwise(F.col("PRCP"))
    )
    .withColumn(
        "WDSP",
        F.when(F.col("WDSP") >= 999, None).otherwise(F.col("WDSP"))
    )
)

weather_clean_df.show(20, truncate=False)

+----------+----+----+----+----+----+------+
|Date      |TEMP|MAX |MIN |PRCP|WDSP|FRSHTT|
+----------+----+----+----+----+----+------+
|2024-01-01|32.7|39.2|30.2|0.0 |3.2 |0     |
|2024-01-02|31.5|44.6|23.0|0.0 |1.5 |100000|
|2024-01-03|36.1|48.2|26.6|0.0 |2.1 |100000|
|2024-01-04|32.5|41.0|26.6|0.0 |5.1 |0     |
|2024-01-05|31.1|33.8|28.4|NULL|7.7 |101000|
|2024-01-06|33.6|35.6|32.0|NULL|4.2 |111000|
|2024-01-07|38.0|48.2|32.0|0.0 |6.6 |0     |
|2024-01-08|38.1|42.8|33.8|NULL|11.5|11000 |
|2024-01-09|34.8|37.4|30.2|NULL|10.7|111000|
|2024-01-10|34.0|53.6|24.8|0.0 |8.8 |0     |
|2024-01-11|42.0|55.4|32.0|0.0 |6.7 |0     |
|2024-01-12|36.0|51.8|17.6|NULL|11.6|111010|
|2024-01-13|18.3|24.8|12.2|NULL|8.6 |1000  |
|2024-01-14|2.7 |15.8|-4.0|NULL|7.3 |1000  |
|2024-01-15|1.9 |10.4|-4.0|NULL|4.1 |1000  |
|2024-01-16|4.5 |14.0|-4.0|NULL|7.8 |1000  |
|2024-01-17|17.2|39.2|6.8 |0.0 |5.6 |0     |
|2024-01-18|30.7|41.0|23.0|0.0 |4.6 |0     |
|2024-01-19|22.5|37.4|10.4|NULL|8.4 |1000  |
|2024-01-2

In [18]:
weather_clean_df.groupBy("Date") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+----+-----+
|Date|count|
+----+-----+
+----+-----+



In [19]:
weather_clean_df.select(
    F.min("Date").alias("Weather_Start"),
    F.max("Date").alias("Weather_End"),
    F.countDistinct("Date").alias("Weather_Days")
).show()

+-------------+-----------+------------+
|Weather_Start|Weather_End|Weather_Days|
+-------------+-----------+------------+
|   2024-01-01| 2024-12-31|         306|
+-------------+-----------+------------+



In [20]:
final_joined_df = joined_df.join(
    weather_clean_df,
    on="Date",
    how="left"
)

In [21]:
print("Rows before weather join:", joined_df.count())
print("Rows after weather join:", final_joined_df.count())

Rows before weather join: 91250
Rows after weather join: 91250


In [23]:
final_joined_df.select(
    F.sum(F.col("TEMP").isNotNull().cast("int")).alias("Rows_With_Weather"),
    F.sum(F.col("TEMP").isNull().cast("int")).alias("Rows_Without_Weather")
).show()

+-----------------+--------------------+
|Rows_With_Weather|Rows_Without_Weather|
+-----------------+--------------------+
|            76250|               15000|
+-----------------+--------------------+



In [24]:
final_joined_df.select(
    "Date",
    "SKU_ID",
    "Warehouse_ID",
    "Units_Sold",
    "Is_Holiday",
    "official_name",
    "TEMP",
    "MAX",
    "MIN",
    "PRCP",
    "WDSP"
).show(20, truncate=False)

+----------+------+------------+----------+----------+-----------------------------------+----+----+----+----+----+
|Date      |SKU_ID|Warehouse_ID|Units_Sold|Is_Holiday|official_name                      |TEMP|MAX |MIN |PRCP|WDSP|
+----------+------+------------+----------+----------+-----------------------------------+----+----+----+----+----+
|2024-01-01|SKU_1 |WH_1        |10        |1         |New Year's Day                     |32.7|39.2|30.2|0.0 |3.2 |
|2024-01-02|SKU_1 |WH_1        |17        |0         |Non-Holiday                        |31.5|44.6|23.0|0.0 |1.5 |
|2024-01-03|SKU_1 |WH_1        |35        |0         |Non-Holiday                        |36.1|48.2|26.6|0.0 |2.1 |
|2024-01-04|SKU_1 |WH_1        |24        |0         |Non-Holiday                        |32.5|41.0|26.6|0.0 |5.1 |
|2024-01-05|SKU_1 |WH_1        |21        |0         |Non-Holiday                        |31.1|33.8|28.4|NULL|7.7 |
|2024-01-06|SKU_1 |WH_1        |18        |0         |Non-Holiday       

In [25]:
output_path = "/home/jovyan/work/CS675_Supply_Chain_Project/data/processed/final_joined"

final_joined_df.write \
    .mode("overwrite") \
    .parquet(output_path)

In [26]:
check_df = spark.read.parquet(output_path)

print("Saved rows:", check_df.count())
print("Saved columns:", len(check_df.columns))

Saved rows: 91250
Saved columns: 25
